# Document parsing with HunyuanOCR and OpenVINO

[HunyuanOCR](https://huggingface.co/tencent/HunyuanOCR) is a lightweight, end-to-end
OCR-specialised vision-language model released by the Tencent Hunyuan team. It pairs a
SigLIP-style vision encoder with a compact Hunyuan text decoder and unifies **document
parsing**, **text spotting**, **information extraction**, and **text-image translation**
in a single end-to-end VLM, while staying small enough for on-device deployment.

In this tutorial we convert HunyuanOCR to OpenVINO IR via optimum-intel, optionally
compress its weights to **INT8** (or keep **FP16**) with [NNCF](https://github.com/openvinotoolkit/nncf),
and run inference on Intel CPU / integrated GPU / Arc GPU. The basic inference usage is
aligned with the official HuggingFace `transformers` (native) snippet from the
[model card](https://huggingface.co/tencent/HunyuanOCR). A streaming Gradio demo — with
the same task presets as the official [HunyuanOCR Space](https://huggingface.co/spaces/tencent/HunyuanOCR) —
is provided as well.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
  - [Select precision](#Select-precision)
- [Run OpenVINO model](#Run-OpenVINO-model)
  - [Select inference device](#Select-inference-device)
- [OCR inference](#OCR-inference)
- [Interactive demo](#Interactive-demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is
using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/hunyuan-ocr/hunyuan-ocr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

HunyuanOCR ships an official `transformers` integration (`HunYuanVLForConditionalGeneration`
+ `AutoProcessor`) that requires `transformers>=5.13`, and relies on the `hunyuan-ocr-support`
branch of the [openvino-dev-samples/optimum-intel](https://github.com/openvino-dev-samples/optimum-intel)
fork for the OpenVINO export.

In [1]:
import requests
from pathlib import Path
from shutil import copyfile

if not Path("cmd_helper.py").exists():
    repo_helper = Path("../../utils/cmd_helper.py")
    if repo_helper.is_file():
        copyfile(repo_helper, "cmd_helper.py")
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
        r.raise_for_status()
        Path("cmd_helper.py").write_text(r.text, encoding="utf-8")

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    Path("notebook_utils.py").write_text(r.text, encoding="utf-8")

# gradio_helper.py ships alongside this notebook; no download needed when
# running from the repository checkout.

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("hunyuan-ocr.ipynb")

In [2]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx

import subprocess
import sys

from notebook_utils import pip_install
from cmd_helper import clone_repo

pip_install("-q", "torch>=2.8", "torchvision", "--extra-index-url", "https://download.pytorch.org/whl/cpu")
pip_install("-q", "transformers<5")
fork_revision = "4085f5e80fb92c6c0eecd0c7f09bcb9d5fe07b14"
fork_dir = Path("optimum-intel-fork")
if not fork_dir.exists():
    import tarfile, urllib.request
    with urllib.request.urlopen("https://codeload.github.com/openvino-dev-samples/optimum-intel/tar.gz/" + fork_revision) as r, tarfile.open(fileobj=r, mode="r|gz") as tf:
        tf.extractall(".")
    fork_dir = Path("optimum-intel-" + fork_revision)
actual_revision = fork_revision
if actual_revision != fork_revision:
    raise RuntimeError(f"Expected {fork_revision} in {fork_dir}, got {actual_revision}")
setup_path = fork_dir / "setup.py"
setup_text = setup_path.read_text(encoding="utf-8")
# Pin Optimum before its Hub requirement changed; allow safetensors 0.8 for hunyuan_vl.
dependencies = {
    '"optimum@git+https://github.com/huggingface/optimum.git"': '"optimum@https://codeload.github.com/huggingface/optimum/tar.gz/52367da7f2d227d82db26d0de10f44c297b8c1eb"',
    '"safetensors<0.8.0"': '"safetensors>=0.8.0,<0.9.0"',
}
for original, pinned in dependencies.items():
    if setup_text.count(original) == 1:
        setup_text = setup_text.replace(original, pinned)
    elif setup_text.count(pinned) != 1:
        raise ValueError(f"Unexpected dependency in {setup_path}: {original}")
setup_path.write_text(setup_text, encoding="utf-8")
# The fork supports transformers<5.14; this revision adds hunyuan_vl and still satisfies that bound.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        str(fork_dir.resolve()),
        "git+https://github.com/huggingface/transformers.git@5d063058dc2ddae5e31e4fab1985701d5cf50133",
    ],
    check=True,
)
pip_install("-q", "openvino>=2026.2.0", "nncf>=2.17.0")
pip_install("-q", "gradio>=5.25.0", "Pillow>=10")

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-rvii6ga_ did not run successfully.
  │ exit code: 128
  ╰─> [8 lines of output]
      [git-wrapper] cloning GitHub repo via gh-proxy.org
      fatal: unable to access 'https://gh-proxy.org/https://github.com/huggingface/transformers.git/': CONNECT tunnel failed, response 403
      [git-wrapper] gh-proxy clone failed, retrying via http://github.com
      error: RPC failed; curl 56 GnuTLS recv error (-9): Error decoding the received TLS packet.
      error: 4675 bytes of body are still expected
      fetch-pack: unexpected disconnect while reading sideband packet
      fatal: early EOF
      fatal: fetch-pack: invalid index-pack output
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'git+https://github.com/huggingface/transformers.git@5d063058dc2ddae5e31e4

CalledProcessError: Command '['/workspace/.venvs/cpu/0e3c03443623f289/bin/python', '-m', 'pip', 'install', '-q', '/workspace/results/hunyuan-ocr/workdir-cpu/optimum-intel-4085f5e80fb92c6c0eecd0c7f09bcb9d5fe07b14', 'git+https://github.com/huggingface/transformers.git@5d063058dc2ddae5e31e4fab1985701d5cf50133']' returned non-zero exit status 1.

In [ ]:
# The demo/sample document image is generated on the fly (no binary asset is
# shipped with the notebook), so this notebook is fully self-contained.
from gradio_helper import make_sample_image

sample_image = make_sample_image()
print("Sample image saved to:", sample_image)

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

[Optimum Intel](https://huggingface.co/docs/optimum/intel/index) exposes a command-line
interface to export HuggingFace models to OpenVINO IR. For HunyuanOCR we use the
`image-text-to-text` task, which produces three IR sub-models: the vision embeddings
(patch embedder + vision transformer + merger), the text-embedding lookup, and the
stateful language-model decoder with the tied LM head.

```bash
optimum-cli export openvino --model tencent/HunyuanOCR --task image-text-to-text --weight-format fp32 HunyuanOCR/FP32
```

### Select precision
[back to top ⬆️](#Table-of-contents:)

So the precision option below controls only the **language model**: **INT8** weight-compresses
the language model and token embeddings (vision stays FP32) — the recommended trade-off — while
**FP32** keeps every sub-model in full precision. Pick a precision below.


In [ ]:
import ipywidgets as widgets

model_id = "tencent/HunyuanOCR"

# The precision below controls only the language model - the vision encoder is always exported in
# FP32 (see the note above):
#   * INT8 - language model + token embeddings weight-compressed to INT8, vision encoder FP32 (recommended)
#   * FP32 - every sub-model in FP32 (largest, reference precision)
precision = widgets.Dropdown(
    options=["INT8", "FP32"],
    value="INT8",
    description="Precision:",
    disabled=False,
)
precision

In [ ]:
from pathlib import Path

import openvino as ov
import nncf
from cmd_helper import optimum_cli

model_base_dir = Path(model_id.split("/")[-1])  # "HunyuanOCR"
model_dir = model_base_dir / precision.value

# SigLIP-style vision encoder of HunyuanOCR produces very large activation magnitudes, which make its
# self-attention numerically hyper-sensitive. OpenVINO's fp16/int8 weight-compressed compute path
# loses too much precision there and yields garbage OCR, so the vision encoder is ALWAYS kept in FP32.
# We therefore export every sub-model in FP32 first and then - for the INT8 preset - weight-compress
# ONLY the language model and token-embedding sub-models to INT8. The language model dominates the
# footprint, so this keeps most of the size benefit while preserving OCR accuracy.
if not model_dir.exists():
    optimum_cli(
        model_id,
        model_dir,
        additional_args={"task": "image-text-to-text", "weight-format": "fp32"},
    )

    if precision.value == "INT8":
        for name in ("openvino_language_model", "openvino_text_embeddings_model"):
            xml_path = model_dir / f"{name}.xml"
            compressed = nncf.compress_weights(ov.Core().read_model(xml_path), mode=nncf.CompressWeightsMode.INT8_ASYM)
            # save to a temporary path first: the source .bin is memory-mapped and cannot be
            # overwritten in place, so we write beside it and then atomically replace.
            tmp_path = model_dir / f"{name}.int8.xml"
            ov.save_model(compressed, tmp_path)
            del compressed
            tmp_path.replace(xml_path)
            tmp_path.with_suffix(".bin").replace(xml_path.with_suffix(".bin"))

print("OpenVINO IR ready at:", model_dir)

## Run OpenVINO model
[back to top ⬆️](#Table-of-contents:)

optimum-intel's `OVModelForVisualCausalLM` loads the IR we just exported and exposes a
transformers-compatible API: `.generate()`, HuggingFace chat templates, token streamers, etc.
The usage below mirrors the native `transformers` snippet from the HunyuanOCR model card.

### Select inference device
[back to top ⬆️](#Table-of-contents:)

HunyuanOCR runs on Intel CPU, integrated GPUs and Arc GPUs. NPU inference is not currently
supported for image-text-to-text models and is excluded from the widget below.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
from transformers import AutoProcessor
from optimum.intel import OVModelForVisualCausalLM

processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device.value)

## OCR inference
[back to top ⬆️](#Table-of-contents:)

HunyuanOCR is driven by task-specific text prompts. The default **document parsing** prompt
extracts the body text as Markdown (tables as HTML, formulas as LaTeX) in reading order.
The code below mirrors the official snippet in the
[`tencent/HunyuanOCR` model card](https://huggingface.co/tencent/HunyuanOCR): build a chat
message with an image + text prompt, apply the chat template, and `generate` greedily.

In [ ]:
import sys
from pathlib import Path
from threading import Thread

from PIL import Image
from IPython.display import display
from transformers import TextIteratorStreamer

# Default HunyuanOCR document-parsing prompt.
DOC_PARSE_PROMPT = (
    "Extract all the body text from the document image in markdown format. "
    "Ignore headers and footers, express tables in HTML format and formulas "
    "in LaTeX format, and organize the output in reading order."
)


def run_hunyuan_ocr(image_path, prompt=DOC_PARSE_PROMPT, max_new_tokens=1024):
    """Run HunyuanOCR and stream the decoded text to stdout as it arrives.

    Returns the full decoded string once generation finishes.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path)},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        streamer=streamer,
    )
    thread = Thread(target=model.generate, kwargs=gen_kwargs, daemon=True)
    thread.start()

    output = ""
    for piece in streamer:
        sys.stdout.write(piece)
        sys.stdout.flush()
        output += piece
    thread.join(timeout=1.0)
    sys.stdout.write("\n")
    sys.stdout.flush()
    return output

In [ ]:
display(Image.open(sample_image))
_ = run_hunyuan_ocr(sample_image, DOC_PARSE_PROMPT)

In [ ]:
# The same model handles other OCR tasks by simply changing the prompt.
# Here we ask it to detect and localise the text (text spotting).
_ = run_hunyuan_ocr(sample_image, "Detect and recognize the text in the image, and output the text with its coordinates.", max_new_tokens=512)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Upload an image, pick a task preset (or type your own instruction), and watch the decoded
text stream in. The task presets mirror the official
[HunyuanOCR Space](https://huggingface.co/spaces/tencent/HunyuanOCR).